# Baselines y seis candidatos: comparación por el criterio del negocio

Ejecutar con Restart / Run All después de completar las fases `modelos` y `ablacion` del coordinador. Las salidas guardadas y el manifiesto de verificación acreditan la ejecución; una plantilla sin salidas sigue pendiente. Se importan el Pipeline y los módulos versionados; las búsquedas ya registradas no se repiten desde el notebook.

La selección prioriza la media entre folds de cobertura de Fail con cupo del 10 % por edición. AP es la métrica técnica común; Brier y log-loss diagnostican calibración. La desviación entre folds es dispersión descriptiva, no un intervalo de confianza.


In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
here = Path.cwd().resolve()
ROOT = next((p for p in [here, *here.parents] if (p/'config/protocolo-v2.json').is_file()), None)
if ROOT is None and (here/'pset-03/v2/config/protocolo-v2.json').is_file():
    ROOT = here/'pset-03/v2'
assert ROOT is not None, 'Abrir el notebook desde v2/notebooks o desde la raíz del repositorio'
os.environ['PSET3_V2_ROOT'] = str(ROOT)
sys.path.insert(0, str(ROOT))
from src.seguimiento import protocolo
from src.common import sha256
plan = protocolo()
def report(relative):
    path = ROOT/relative
    if not path.is_file():
        raise RuntimeError('Etapa pendiente: falta '+relative+'. Revisar docs/estado-v2.json y esperar al coordinador; no sustituir por resultados de v1.')
    return json.loads(path.read_text())
print('Protocolo verificado:', plan['id'])
print('Este notebook consulta resultados reales y reproduce sus cálculos; no inicia nuevas búsquedas ni llamadas de inferencia API.')


In [ ]:
models = report('docs/modelos/resultados.json')
assert models['status'] == 'complete' and models['protocol_id'] == plan['id']
assert len(models['baselines']) == 2 and len(models['candidates']) == 6
from src.modelos import candidatos
rows = []
for result in models['candidates']:
    assert len(result['folds']) == 5
    np.testing.assert_allclose(result['cv_coverage_mean'], np.mean([f['cv_coverage'] for f in result['folds']]))
    estimator = candidatos()[result['family']]
    params = estimator.get_params(deep=False)
    rows.append({'familia':result['family'], 'AP_train_muestra':result['train_ap_fail_mean'],
                 'AP_train_DE':float(np.std([f['train_ap_fail'] for f in result['folds']], ddof=1)),
                 'AP_CV_media':result['cv_ap_fail_mean'], 'AP_CV_DE':result['cv_ap_fail_std'],
                 'brecha_AP':result['ap_gap'], 'cobertura_CV_media':result['cv_coverage_mean'],
                 'cobertura_CV_DE':result['cv_coverage_std'], 'Brier_CV':result['cv_brier_fail_mean'],
                 'logloss_CV':result['cv_logloss_mean'], 'fit_CV_seg':result['cv_fit_seconds_total'],
                 'inferencia_CV_seg':result['cv_inference_seconds_total'],
                 'API_costo_nuevo_USD':result.get('api_cost_new_calls_usd',0),
                 'learning_rate':params.get('learning_rate_init',params.get('learning_rate','no aplica')),
                 'run_id':result['run_id']})
comparison = pd.DataFrame(rows).sort_values('cobertura_CV_media',ascending=False)
display(comparison)
display(pd.DataFrame([{'baseline':'Dummy más frecuente', 'cobertura_CV':models['baselines'][0]['cv_coverage_mean'], 'AP_CV':models['baselines'][0]['cv_ap_fail_mean']},
                      {'baseline':'Reactiva al cierre', 'cobertura_CV':models['baselines'][1]['preventive_coverage'], 'AP_CV':np.nan}]))


## Cómo interpretar entrenamiento, recursos y sobreajuste

La métrica de entrenamiento usa exactamente la misma muestra estratificada de hasta 1.000 filas por train-fold para los seis candidatos. Se identifica como muestra y no como la métrica de todo el entrenamiento. La validación incluye todas las mismas matrículas y folds para los seis.

El GPT usa 18 ejemplos de train-fold y sus proporciones de clase; no ajusta sus pesos con todas las filas. Su fit construye el contexto y su mayor costo ocurre al predecir. Se reportan inferencia, latencia y consumo además del tiempo de fit. El límite global incluye pilotos e intentos interrumpidos; el costo nuevo del run no equivale al consumo total del proyecto. La reproducción exacta usa respuestas archivadas y no presupone determinismo del proveedor.


In [ ]:
for row in models['candidates']:
    warnings = [w for fold in row['folds'] for w in fold.get('warnings',[])]
    print(row['family'], '| brecha AP:',round(row['ap_gap'],4), '| avisos registrados:',len(warnings))
    if warnings:
        display(pd.DataFrame(warnings).drop_duplicates().head(5))


## Aporte de las variables derivadas

La ablación compara una arquitectura lineal fija y los mismos cinco folds: estáticas, longitudinal original, solo log1p, solo intensidad normalizada y ambas derivadas. Se conservan los resultados negativos. Una mejora menor que la variación entre folds no se presenta como una mejora demostrada.


In [ ]:
ablation = report('docs/ablacion/resultados.json')
assert ablation['protocol_id'] == plan['id'] and ablation['test_used'] is False
display(pd.DataFrame([{k:v for k,v in r.items() if k!='folds'} for r in ablation['results']]))
